# 🚨 Incident Commander — RL Training Notebook

**OpenEnv Hackathon 2025**

This notebook trains a Qwen2.5 language model to act as an AI Site Reliability Engineer (SRE) that diagnoses and resolves production microservice outages using a **two-phase pipeline**:

1. **SFT Warm-Start** — Teach the model valid JSON output format + action priors via expert demonstrations
2. **GRPO Fine-Tuning** — Reinforcement learning with direct-action reward shaping

| Agent | Avg Score |
|---|---|
| Random | 0.40 |
| Heuristic baseline | 0.80 |
| **Our trained model** | **0.84+** |

**Runtime:** ~30-40 min on T4 GPU (free Colab) · ~15 min on L40S

---
**GitHub:** https://github.com/hs-zz27/incident-commander-openenv  
**HuggingFace Space:** https://huggingface.co/spaces/hs-zz27/incident-commander

## 📦 Step 1: Environment Setup

In [ ]:
# Verify GPU is available
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print(f'✅ GPU: {result.stdout.strip()}')
else:
    print('⚠️  No GPU found. Go to Runtime → Change runtime type → GPU')

In [ ]:
import os

REPO_URL = 'https://github.com/hs-zz27/incident-commander-openenv.git'
REPO_DIR = 'incident-commander-openenv'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
    print('✅ Repo cloned')
else:
    !cd {REPO_DIR} && git pull
    print('✅ Repo updated')

%cd {REPO_DIR}

In [ ]:
# Install project + training dependencies
!pip install -e '.[train]' -q
print('✅ Dependencies installed')

In [ ]:
# Verify environment works
import sys
sys.path.insert(0, '.')

from server.environment import IncidentCommanderEnvironment
from server.models import IncidentAction, ActionType

env = IncidentCommanderEnvironment()
obs = env.reset(task_name='single_service_failure', seed=42)
print(f'✅ Environment OK — task: {obs.task_name}, services: {list(obs.services.keys())}')
print(f'   System health: {obs.system_health_score:.2%}')

## 📊 Step 2: Run Baselines (Random vs Heuristic)

In [ ]:
# Run baseline evaluation (random + heuristic agents)
# This takes ~2 minutes and produces the data for our comparison chart
!python run_baselines.py --episodes 10
print('✅ Baselines complete — results saved to results/baseline_rewards.json')

## 🧠 Step 3: SFT Warm-Start

Before GRPO, we do supervised fine-tuning on **expert demonstrations**.

**Why?** Without SFT, a 0.5B model produces mostly unparseable outputs, giving GRPO a flat reward landscape and causing mode collapse.

The SFT phase teaches:
- ✅ Output format — valid JSON `{"action_type": "...", "service_name": "..."}`
- ✅ Action priors — inspect before fix, dependency ordering, rollback bad versions

In [ ]:
# Configuration — adjust based on your GPU
MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'   # swap to 1.5B if you have an A100/L40S
SFT_ADAPTER_DIR = 'sft_adapter_v5'
SFT_MERGED_DIR  = 'sft_merged_v5'

!python sft_warmstart.py \
    --model {MODEL} \
    --epochs 3 --batch-size 4 --lr 2e-5 --num-seeds 8 \
    --use-lora --use-4bit --gradient-checkpointing \
    --output-dir {SFT_ADAPTER_DIR} \
    --merged-output-dir {SFT_MERGED_DIR}

print(f'✅ SFT complete — merged model saved to {SFT_MERGED_DIR}/')

In [ ]:
# Plot SFT training loss curve
import json, matplotlib.pyplot as plt

with open('results/sft_training_log.json') as f:
    sft_log = json.load(f)

log_history = sft_log.get('log_history', [])
steps = [e['step'] for e in log_history if 'loss' in e]
losses = [e['loss'] for e in log_history if 'loss' in e]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(steps, losses, color='#6366f1', linewidth=2)
ax.fill_between(steps, losses, alpha=0.15, color='#6366f1')
ax.set_xlabel('Step', fontsize=12)
ax.set_ylabel('Training Loss', fontsize=12)
ax.set_title('SFT Warm-Start — Training Loss', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('results/sft_loss_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Final SFT loss: {losses[-1]:.4f}')

## 🎯 Step 4: GRPO Reinforcement Learning

**Group Relative Policy Optimization (GRPO)** fine-tunes the model using the environment's reward signal.

Our reward function scores each action based on:
- 🩺 **Health delta** — did system health go up after this action?
- 🎯 **Correct recovery** — did the model restart the right service?
- ⏱️ **Efficiency** — fewer steps = higher score
- 📋 **Memory** — did the model write a runbook after resolution?

In [ ]:
GRPO_OUTPUT_DIR = 'trained_model_v5'

!python train_grpo.py \
    --model {SFT_MERGED_DIR} \
    --steps 200 --batch-size 1 --lr 3e-6 \
    --num-generations 8 --num-seeds 8 \
    --use-lora --gradient-checkpointing \
    --lora-r 16 --lora-alpha 32 \
    --save-steps 25 --log-every 10 \
    --reward-mode direct \
    --snapshot-steps '1,2,3,4,5,6' \
    --output-dir {GRPO_OUTPUT_DIR}

print(f'✅ GRPO complete — adapter saved to {GRPO_OUTPUT_DIR}/')

In [ ]:
# Plot GRPO reward curve
import json, matplotlib.pyplot as plt
import numpy as np

with open('results/training_log.json') as f:
    grpo_log = json.load(f)

log_history = grpo_log.get('log_history', [])
rewards = [e.get('rewards/mean', e.get('reward', None)) for e in log_history if e.get('rewards/mean') or e.get('reward')]
steps   = list(range(1, len(rewards)+1))

# Smooth with rolling average
def smooth(y, w=5):
    return np.convolve(y, np.ones(w)/w, mode='valid')

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(steps, rewards, color='#94a3b8', linewidth=1, alpha=0.5, label='Raw')
if len(rewards) > 5:
    smoothed = smooth(rewards)
    ax.plot(range(1, len(smoothed)+1), smoothed, color='#22c55e', linewidth=2.5, label='Smoothed (w=5)')
ax.axhline(y=0, color='#ef4444', linewidth=1, linestyle='--', alpha=0.6, label='Zero reward')
ax.set_xlabel('GRPO Step', fontsize=12)
ax.set_ylabel('Mean Reward', fontsize=12)
ax.set_title('GRPO Training — Mean Reward per Step', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('results/grpo_reward_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Final reward: {rewards[-1]:.4f} | Peak reward: {max(rewards):.4f}')

## 🏆 Step 5: Evaluate Trained Model

In [ ]:
!python evaluate_trained.py \
    --adapter {GRPO_OUTPUT_DIR} \
    --base-model {SFT_MERGED_DIR} \
    --episodes 5 --verbose

In [ ]:
# Final comparison bar chart: Random vs Heuristic vs Trained
import json, matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Load baseline results
with open('results/baseline_rewards.json') as f:
    baseline = json.load(f)

tasks = ['single_service_failure', 'cascading_failure', 'hidden_root_cause']
task_labels = ['Single Service\nFailure', 'Cascading\nFailure', 'Hidden Root\nCause']
agents = ['random', 'heuristic', 'trained']
colors = ['#ef4444', '#f59e0b', '#22c55e']
labels = ['Random Agent', 'Heuristic Baseline', 'Our Trained Model (GRPO)']

x = np.arange(len(tasks))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
for i, (agent, color, label) in enumerate(zip(agents, colors, labels)):
    scores = [baseline.get(agent, {}).get(t, 0) for t in tasks]
    bars = ax.bar(x + i*width, scores, width, label=label, color=color, alpha=0.85, edgecolor='white')
    for bar, score in zip(bars, scores):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                f'{score:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xlabel('Task', fontsize=13)
ax.set_ylabel('Average Score (0–1)', fontsize=13)
ax.set_title('Incident Commander — Agent Performance Comparison', fontsize=15, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(task_labels, fontsize=11)
ax.set_ylim(0, 1.1)
ax.legend(fontsize=10)
ax.grid(True, axis='y', alpha=0.3)
ax.spines[['top','right']].set_visible(False)
ax.axhline(y=0.76, color='#6366f1', linewidth=1.5, linestyle='--', alpha=0.7, label='Hackathon bar')
ax.text(2.7, 0.77, 'Hackathon threshold (0.76)', color='#6366f1', fontsize=9)

plt.tight_layout()
plt.savefig('results/agent_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## ☁️ Step 6: Upload Model to HuggingFace Hub

In [ ]:
from google.colab import userdata

# Store your HF token as a Colab Secret named 'HF_TOKEN'
# (Secrets panel: 🔑 icon in left sidebar)
HF_TOKEN = userdata.get('HF_TOKEN')
HF_REPO  = 'hs-zz27/incident-commander-v5'

!hf auth login --token {HF_TOKEN}

# Upload trained adapter
!hf upload {HF_REPO} ./{GRPO_OUTPUT_DIR} {GRPO_OUTPUT_DIR}

# Upload merged SFT base
!hf upload {HF_REPO} ./{SFT_MERGED_DIR} {SFT_MERGED_DIR}

# Upload plots
!hf upload {HF_REPO} ./results/agent_comparison.png agent_comparison.png
!hf upload {HF_REPO} ./results/grpo_reward_curve.png grpo_reward_curve.png
!hf upload {HF_REPO} ./results/sft_loss_curve.png sft_loss_curve.png

print(f'✅ All artifacts uploaded to https://huggingface.co/{HF_REPO}')

## 📋 Summary

### What we built
An **OpenEnv-compliant RL environment** where a Qwen2.5 language model acts as an AI SRE, diagnosing and resolving production microservice outages.

### 3-Layer Architecture
```
Layer 1: Deterministic Orchestrator  ← Safety net / heuristic override
    ↓
Layer 2: SFT Warm-Start              ← Teach JSON format + action priors
    ↓  
Layer 3: GRPO Fine-Tuning            ← RL with direct-action reward shaping
```

### Reward Signal
- **Health delta** — system health improvement per step
- **Correct recovery** — did the model target the right service?
- **Efficiency bonus** — fewer steps = higher score
- **Memory bonus** — writing a runbook after resolution

### Results
| Agent | Score |
|---|---|
| Random | 0.40 |
| Heuristic | 0.80 |
| **Trained (GRPO)** | **0.84+** |

**Links:**
- 🤗 Space: https://huggingface.co/spaces/hs-zz27/incident-commander
- 💾 Model: https://huggingface.co/hs-zz27/incident-commander-v5
- 📁 Code: https://github.com/hs-zz27/incident-commander-openenv